In [ ]:
import os
import sys
import json
import pickle
import pandas as pd
import numpy as np
import torch
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE, SelectFromModel

# Point to the src folder
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.config import *
from src.model import UnifiedMaterialsRegressor
from src.data_loader import get_loaders
from src.train import train_pipeline

print(f"Environment Ready! Using device: {DEVICE}")

In [ ]:
print("Loading dictionaries from disk...")

with open(SPLIT_PATH, "r") as f: split = json.loads(f.read())
with open(RAW_DIR / "metadata.json", "r") as f:
    id2bg = {m["material_id"]: float(m["band_gap"]) for m in json.loads(f.read()).get("materials", [])}

with open(DATA_DIR / "magpie_all_scaled.pkl", "rb") as f: magpie_scaled = pickle.load(f)
with open(DATA_DIR / "struct_scaled.pkl", "rb") as f: struct_scaled = pickle.load(f)
with open(DATA_DIR / "geo_scaled.pkl", "rb") as f: geo_scaled = pickle.load(f)

# Load XRD files
xrd_scaled = {}
for fname in os.listdir(XRD_DIR):
    if fname.endswith('.npy'):
        mid = fname.split('.')[0]
        xrd_scaled[mid] = np.load(XRD_DIR / fname)

print(f"Data loaded! Train size: {len(split['train'])} | Total XRD patterns: {len(xrd_scaled)}")

In [ ]:
print("Running Feature Selection on Training Set...")

# Extract just the training tabular targets and features
train_mids = [mid for mid in split["train"] if mid in magpie_scaled and mid in id2bg]
X_tr_m1 = np.array([magpie_scaled[m][0] for m in train_mids])
X_tr_m2 = np.array([magpie_scaled[m][1] for m in train_mids])
y_tr_log = np.log1p([id2bg[m] for m in train_mids])

# Fit the selectors
rf_sel = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=SEED, n_jobs=None)

print("-> Fitting RFE...")
rfe1 = RFE(rf_sel, n_features_to_select=24, step=5).fit(X_tr_m1, y_tr_log)
rfe2 = RFE(rf_sel, n_features_to_select=30, step=5).fit(X_tr_m2, y_tr_log)

print("-> Fitting RF Importance...")
sfm1 = SelectFromModel(rf_sel, max_features=24, threshold=-np.inf).fit(X_tr_m1, y_tr_log)
sfm2 = SelectFromModel(rf_sel, max_features=30, threshold=-np.inf).fit(X_tr_m2, y_tr_log)

# Create the dynamic feature dictionaries
mag_full = magpie_scaled
mag_rfe = {mid: (x1[rfe1.support_], x2[rfe2.support_]) for mid, (x1, x2) in magpie_scaled.items()}
mag_rf = {mid: (x1[sfm1.get_support()], x2[sfm2.get_support()]) for mid, (x1, x2) in magpie_scaled.items()}

sample_mid = list(magpie_scaled.keys())[0]
DIM_F1 = len(magpie_scaled[sample_mid][0])
DIM_F2 = len(magpie_scaled[sample_mid][1])

print("Feature selection complete!")

In [ ]:
ablation_tests = {
    
    "1. Multimodal (All Features)": {"c": {"xrd": True,  "tabular": True},  "g1": DIM_F1, "g2": DIM_F2, "mag_source": mag_full},
    "2. Multimodal (Magpie RFE)":   {"c": {"xrd": True,  "tabular": True},  "g1": 24,     "g2": 30,     "mag_source": mag_rfe},
    "3. Multimodal (Magpie RF)":    {"c": {"xrd": True,  "tabular": True},  "g1": 24,     "g2": 30,     "mag_source": mag_rf},
    "4. Tabular Only (All)":        {"c": {"xrd": False, "tabular": True},  "g1": DIM_F1, "g2": DIM_F2, "mag_source": mag_full},
    "5. Tabular Only (Magpie RFE)": {"c": {"xrd": False, "tabular": True},  "g1": 24,     "g2": 30,     "mag_source": mag_rfe},
    "6. Tabular Only (Magpie RF)":  {"c": {"xrd": False, "tabular": True},  "g1": 24,     "g2": 30,     "mag_source": mag_rf},
    "7. XRD Patterns Only":         {"c": {"xrd": True,  "tabular": False}, "g1": DIM_F1, "g2": DIM_F2, "mag_source": mag_full} 

}

results = []

for name, p in ablation_tests.items():
    print(f"\n{'='*50}\n TRAINING: {name}\n{'='*50}")
    
    tr_loader, va_loader, te_loader = get_loaders(split, p["mag_source"], struct_scaled, geo_scaled, xrd_scaled, id2bg)
    model = UnifiedMaterialsRegressor(e_g1=p["g1"], e_g2=p["g2"], active_branches=p["c"], dropout=0.10)
    model = model.to("cuda")
    t_mae, t_rmse, t_r2 = train_pipeline(model, tr_loader, va_loader, te_loader)
    
    results.append({"Experiment": name, "Test MAE (eV)": round(t_mae, 4), "Test RMSE (eV)": round(t_rmse, 4), "Test R2": round(t_r2, 4)})

df_results = pd.DataFrame(results)
df_results.to_csv(RESULTS_DIR / "dl_ablation_results.csv", index=False)
print(f"\nAll tests complete! Saved to {RESULTS_DIR / 'dl_ablation_results.csv'}")
display(df_results.sort_values(by="Test MAE (eV)"))